In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [29]:
torch.cuda.is_available()

True

In [30]:
torch.manual_seed(42)

In [31]:
train = pd.read_csv('https://raw.githubusercontent.com/guilhermedom/cnn-fashion-mnist/main/data/raw/fashion-mnist-train.zip',compression='zip')
test = pd.read_csv('https://raw.githubusercontent.com/guilhermedom/cnn-fashion-mnist/main/data/raw/fashion-mnist-test.zip',compression='zip')

In [32]:
train.shape,test.shape

((60000, 785), (10000, 785))

In [33]:
X_train = train.iloc[:,1:].values
y_train = train.iloc[:,0].values

In [34]:
X_test = test.iloc[:,1:].values
y_test = test.iloc[:,0].values

In [35]:
X_train = X_train/255.0
X_test = X_test/255.0

In [36]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
        self.labels = torch.tensor(labels,dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,idx):
        return self.features[idx],self.labels[idx]

In [37]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [38]:
train_loader = DataLoader(train_dataset,batch_size=32,pin_memory=True,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,pin_memory=True,shuffle=False)

In [39]:
class MyNN(nn.Module):
    def __init__(self,input_features):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_features,32,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(32,64,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(64,10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [40]:
learning_rate = 0.01
ephocs = 100

In [41]:
device = torch.device('cuda')

In [42]:
model = MyNN(1)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=0.01,weight_decay=1e-4)

In [43]:
for epoch in range(ephocs):
    total_epoch_loss = 0
    for batch_features,batch_labels in train_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs,batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss = total_epoch_loss + loss.item()
    
    avg_loss = total_epoch_loss/len(train_loader)
    if (epoch+1) % 10 == 0:
        print(f"Epoch: {epoch+1}, Loss: {avg_loss}")

Epoch: 10, Loss: 0.1824070952743292
Epoch: 20, Loss: 0.11065843119720618
Epoch: 30, Loss: 0.07062262405660004
Epoch: 40, Loss: 0.048703213831111015
Epoch: 50, Loss: 0.03971325230397827
Epoch: 60, Loss: 0.02817265814886972
Epoch: 70, Loss: 0.025586721136257012
Epoch: 80, Loss: 0.021939360739673914
Epoch: 90, Loss: 0.018319552789686472
Epoch: 100, Loss: 0.01593080721808983


In [44]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [50]:
total = 0
correct = 0
with torch.no_grad():
    for batch_features,batch_labels in test_loader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        outputs = model(batch_features)
        _,predicted = torch.max(outputs,1)
        total = total+batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()
print(f"Accuracy: {(correct/total)*100}%")

Accuracy: 93.03%
